# Notebook 3: RegulationQA Main Results

Reproduces Table 3 and Table 4 (per-document) of the paper.

**Systems:** BGE only, BM25 only, Full pipeline

**Metrics:** Token F1, Recall@F1>=0.3, Recall@F1>=0.5

**Dataset:** RegulationQA (n=1,002)

**Runtime:** ~60 min on T4

In [ ]:
!pip install -q sentence-transformers faiss-cpu rank-bm25 nltk

import pickle, json, string, numpy as np, pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from nltk.tokenize import sent_tokenize
import faiss, nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

In [ ]:
# Load RegulationQA artifacts
print('Loading RegulationQA artifacts...')
with open('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/chunks.pkl', 'rb') as f:
    all_chunks = pickle.load(f)
index = faiss.read_index('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/faiss_index.bin')
with open('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/bm25.pkl', 'rb') as f:
    bm25 = pickle.load(f)
rqa_df = pd.read_csv('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/regulationqa_v1.csv')
bge_model     = SentenceTransformer('BAAI/bge-base-en-v1.5')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print(f'Chunks: {len(all_chunks)}, Questions: {len(rqa_df)}')

In [ ]:
# Helper functions
def tokenize(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.split()

def token_f1(prediction, ground_truth):
    pred   = prediction.lower().split()
    gt     = ground_truth.lower().split()
    common = set(pred) & set(gt)
    if not common:
        return 0.0
    p = len(common) / len(pred)
    r = len(common) / len(gt)
    return 2 * p * r / (p + r)

def inner_decay(sentences, ce_scores, threshold=0.0, delta=2.0):
    if not sentences:
        return []
    scored = sorted(zip(ce_scores, sentences), reverse=True)
    selected = []
    for i, (score, sent) in enumerate(scored):
        if score < threshold:
            break
        if i > 0 and (scored[i-1][0] - score) > delta:
            break
        selected.append((score, sent))
    return selected if selected else [scored[0]]

def get_window(doc_name, passage_idx):
    doc_chunks  = [(i, c) for i, c in enumerate(all_chunks) if c['doc_name'] == doc_name]
    current_pos = next((pos for pos, (i, c) in enumerate(doc_chunks) if i == passage_idx), None)
    if current_pos is None:
        return all_chunks[passage_idx]['text']
    window = []
    for offset in [-1, 0, 1]:
        pos = current_pos + offset
        if 0 <= pos < len(doc_chunks):
            window.append(doc_chunks[pos][1]['text'])
    return ' '.join(window)

def get_rrf_candidates(query, top_n=100, rrf_k=60):
    qe = bge_model.encode([query], normalize_embeddings=True).astype(np.float32)
    _, bge_indices = index.search(qe, top_n)
    bge_indices    = bge_indices[0]
    bm25_scores    = bm25.get_scores(tokenize(query))
    bm25_indices   = np.argsort(bm25_scores)[::-1][:top_n]
    rrf_scores = {}
    for rank, idx in enumerate(bge_indices):
        c   = all_chunks[idx]
        pid = (c['doc_name'], c['chunk_idx'])
        rrf_scores[pid] = rrf_scores.get(pid, 0) + 1/(rrf_k + rank + 1)
    for rank, idx in enumerate(bm25_indices):
        c   = all_chunks[idx]
        pid = (c['doc_name'], c['chunk_idx'])
        rrf_scores[pid] = rrf_scores.get(pid, 0) + 1/(rrf_k + rank + 1)
    sorted_pids = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:20]
    pid_to_idx = {}
    for i, c in enumerate(all_chunks):
        pid = (c['doc_name'], c['chunk_idx'])
        if pid not in pid_to_idx:
            pid_to_idx[pid] = i
    return sorted_pids, pid_to_idx

print('Helpers ready.')

In [ ]:
# Retrieval functions
def retrieve_bge_only_rqa(query, top_n=10):
    qe = bge_model.encode([query], normalize_embeddings=True).astype(np.float32)
    _, indices = index.search(qe, top_n)
    results, seen = [], set()
    for idx in indices[0]:
        if idx in seen:
            continue
        seen.add(idx)
        c     = all_chunks[idx]
        sents = sent_tokenize(c['text'])
        if sents:
            results.append({'doc_name': c['doc_name'], 'extracted_spans': sents[:3]})
    return results

def retrieve_bm25_only_rqa(query, top_n=10):
    tokens      = tokenize(query)
    bm25_scores = bm25.get_scores(tokens)
    top_indices = np.argsort(bm25_scores)[::-1][:top_n]
    results, seen = [], set()
    for idx in top_indices:
        if idx in seen:
            continue
        seen.add(idx)
        c     = all_chunks[idx]
        sents = sent_tokenize(c['text'])
        if sents:
            results.append({'doc_name': c['doc_name'], 'extracted_spans': sents[:3]})
    return results

def retrieve_full_pipeline_rqa(query, top_n=100):
    sorted_pids, pid_to_idx = get_rrf_candidates(query, top_n)
    results, seen = [], set()
    for pid, _ in sorted_pids:
        if pid in seen:
            continue
        seen.add(pid)
        idx = pid_to_idx.get(pid)
        if idx is None:
            continue
        c          = all_chunks[idx]
        curr_sents = sent_tokenize(c['text'])
        if not curr_sents:
            continue
        curr_ce  = cross_encoder.predict([[query, s] for s in curr_sents])
        top_ce   = float(max(curr_ce))
        window   = get_window(c['doc_name'], idx)
        win_sents = sent_tokenize(window)
        win_ce    = cross_encoder.predict([[query, s] for s in win_sents])
        kept      = inner_decay(win_sents, win_ce)
        if kept:
            results.append({'doc_name': c['doc_name'], 'top_ce_score': top_ce,
                            'extracted_spans': [s for _, s in kept]})
    results.sort(key=lambda x: x['top_ce_score'], reverse=True)
    return results

print('Retrieval functions ready.')

In [ ]:
def evaluate_rqa(df, retrieve_fn, top_n=100):
    f1_scores, r03, r05 = [], [], []
    results_log = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        query        = row['question']
        ground_truth = str(row['ground_truth_answer'])
        doc_source   = row['document_source']
        results      = retrieve_fn(query, top_n=top_n)
        best_f1, best_span, best_doc = 0.0, '', ''
        for r in results[:3]:
            for span in r.get('extracted_spans', []):
                f1 = token_f1(span, ground_truth)
                if f1 > best_f1:
                    best_f1, best_span = f1, span
                    best_doc = r.get('doc_name', '')
        f1_scores.append(best_f1)
        r03.append(1.0 if best_f1 >= 0.3 else 0.0)
        r05.append(1.0 if best_f1 >= 0.5 else 0.0)
        results_log.append({'question': query, 'doc_source': doc_source,
                            'ground_truth': ground_truth, 'best_span': best_span,
                            'best_doc': best_doc, 'f1': best_f1})
    return {'Mean F1': np.mean(f1_scores), 'Recall@0.3': np.mean(r03),
            'Recall@0.5': np.mean(r05), 'results_log': results_log}

print('Evaluator ready.')

In [ ]:
# Run all systems
all_results = {}

print('1/3 BGE only...')
all_results['BGE only'] = evaluate_rqa(rqa_df, retrieve_bge_only_rqa)
print(f"  F1={all_results['BGE only']['Mean F1']:.4f}")

print('\n2/3 BM25 only...')
all_results['BM25 only'] = evaluate_rqa(rqa_df, retrieve_bm25_only_rqa)
print(f"  F1={all_results['BM25 only']['Mean F1']:.4f}")

print('\n3/3 Full pipeline...')
all_results['Full pipeline'] = evaluate_rqa(rqa_df, retrieve_full_pipeline_rqa)
print(f"  F1={all_results['Full pipeline']['Mean F1']:.4f}")

In [ ]:
# Table 3
print('\n=== TABLE 3: RegulationQA Results (n=1,002) ===')
print(f'{"System":<20} {"F1":>7} {"R@0.3":>7} {"R@0.5":>7}')
print('-' * 45)
for system, metrics in all_results.items():
    print(f'{system:<20} {metrics["Mean F1"]:>7.4f} {metrics["Recall@0.3"]:>7.4f} {metrics["Recall@0.5"]:>7.4f}')

In [ ]:
import pandas as pd
# Table 4 — per-document breakdown
log_df = pd.DataFrame(all_results['Full pipeline']['results_log'])
print('\n=== TABLE 4: Per-Document F1 (Full Pipeline) ===')
doc_f1 = log_df.groupby('doc_source')['f1'].agg(['mean', 'count']).sort_values('mean', ascending=False)
print(doc_f1.to_string())

In [ ]:
# Save
results_to_save = {k: {m: v for m, v in metrics.items() if m != 'results_log'}
                   for k, metrics in all_results.items()}
with open('/kaggle/working/regulationqa_main_results.json', 'w') as f:
    json.dump(results_to_save, f, indent=2)
log_df.to_csv('/kaggle/working/regulationqa_extraction_results.csv', index=False)
print('Saved: regulationqa_main_results.json, regulationqa_extraction_results.csv')